# PyTorch를 이용한 선형 회귀 모델 학습

- nn.Module, F.mse_loss, 그리고 optim.SGD와 같은 PyTorch의 내장 함수 를 사용하여 코드의 간결성을 높이고, 모델의 학습과 예측 과정을 자동화합니다.

- 최적화 알고리즘 모듈인 torch.optim을 optim이라는 이름으로 가져옵니다.
- optim.SGD의 첫 번째 인자로는 모델의 매개변수를 넘겨주어야 합니다. model.parameters()를 사용하여 모델의 모든 매개변수를 최적화기에 전달합니다.
- F.mse_loss 함수의 첫 번째 인자와 두 번째 인자로는 각각 모델의 예측값(pred)과 실제 타겟값(y_train)을 넣어줍니다.

In [1]:
import torch 
import torch.nn as nn
torch.manual_seed(42) # 난수 코드 실행간 난수의 일관된 결과를 얻기 위해 

x_train = torch.tensor([[1], [2], [3]], dtype=torch.float)
y_train = torch.tensor([[3], [6], [9]], dtype=torch.float)

in_features = x_train.shape[1]
out_features = y_train.shape[1]

model = nn.Linear(in_features=in_features, out_features=out_features)

import torch.optim as optim
import torch.nn.functional as F

optimizer = optim.SGD(model.parameters(), lr=0.01)

nb_epochs = 1000  # 반복 횟수 설정
for epoch in range(nb_epochs):

    pred = model(x_train)

    loss = F.mse_loss(pred, y_train)

    optimizer.zero_grad()  
    loss.backward()        
    optimizer.step()      
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1:4d}/{nb_epochs}] W: {model.weight.item():.3f}, b: {model.bias.item():.3f} loss: {loss.item():.3f}')

Epoch [ 100/1000] W: 2.475, b: 1.194 loss: 0.206
Epoch [ 200/1000] W: 2.587, b: 0.939 loss: 0.127
Epoch [ 300/1000] W: 2.675, b: 0.738 loss: 0.079
Epoch [ 400/1000] W: 2.745, b: 0.580 loss: 0.049
Epoch [ 500/1000] W: 2.799, b: 0.456 loss: 0.030
Epoch [ 600/1000] W: 2.842, b: 0.359 loss: 0.019
Epoch [ 700/1000] W: 2.876, b: 0.282 loss: 0.011
Epoch [ 800/1000] W: 2.903, b: 0.222 loss: 0.007
Epoch [ 900/1000] W: 2.923, b: 0.174 loss: 0.004
Epoch [1000/1000] W: 2.940, b: 0.137 loss: 0.003


In [2]:
test_x = torch.tensor([[10]], dtype=torch.float)

with torch.no_grad():
    pred_y = model(test_x)
    print("훈련 후 입력이 10일 때의 예측값 :", pred_y.item())       

훈련 후 입력이 10일 때의 예측값 : 29.534687042236328


# nn.Module을 상속으로 구현한 사용자 정의 선형 회귀 클래스 

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

x_train = torch.tensor([[1], [2], [3]], dtype=torch.float)
y_train = torch.tensor([[3], [6], [9]], dtype=torch.float)

class MyLinearModel(nn.Module):
    def __init__(self):
        super(MyLinearModel, self).__init__()
        # 선형 레이어 정의, in_features와 out_features를 1로 고정
        self.linear = nn.Linear(in_features=1, out_features=1)
        
    def forward(self, x):
        # 입력 x에 대해 선형 변환 수행
        return self.linear(x)

# 인자 없이 모델 생성
model = MyLinearModel()

## super()의 의미와 역할

### super()가 뭔가요?

`super(MyLinearModel, self).__init__()`은 **부모 클래스(nn.Module)의 `__init__` 메서드를 호출**합니다.

즉, "우리가 상속받은 부모의 초기화 작업을 먼저 해주세요"라는 뜻입니다.

---

### 왜 꼭 필요한가?

`nn.Module`의 부모 클래스 초기화에서는 중요한 설정들이 이루어집니다:

- 모델의 매개변수 관리 시스템 초기화
- 학습/평가 모드 설정 시스템 준비
- 역전파(backpropagation) 메커니즘 초기화
- 디바이스 관리 시스템 설정

만약 `super().__init__()`을 호출하지 않으면 이 모든 기능이 제대로 작동하지 않습니다.

---

### 직관적인 비유

```
자동차 클래스를 상속받아 전기차 클래스를 만드는 상황:

자동차.__init__(): 엔진, 휠, 배터리 기본 시스템 초기화
전기차.__init__(): super() 호출 → 자동차의 기본 설정 먼저 수행
            → 그 다음에 전기차만의 특수 설정 추가
```

`super().__init__()` 없이는 자동차의 기본 시스템이 설정되지 않아, 전기차도 제대로 작동할 수 없습니다.

---

### 코드로 보는 비교

아래 코드를 실행해서 `super().__init__()`의 필요성을 확인해보세요.

In [7]:
import torch
import torch.nn as nn

# ─────────────────────────────────────────────────────
# Case 1: super().__init__() ✓ 올바른 방법
# ─────────────────────────────────────────────────────
class CorrectModel(nn.Module):
    def __init__(self):
        super(CorrectModel, self).__init__()  # 부모 클래스 초기화 ✓
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

# ─────────────────────────────────────────────────────
# Case 2: super().__init__() 없이 ✗ 잘못된 방법
# ─────────────────────────────────────────────────────
class IncorrectModel(nn.Module):
    def __init__(self):
        # super().__init__() 호출 안 함 ✗
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

# 테스트
print("=" * 60)
print("[ 올바른 모델 ]")
print("=" * 60)
correct_model = CorrectModel()
print(f"✓ 모델 생성 성공")
print(f"매개변수 목록: {list(correct_model.parameters())}")
print(f"model.parameters() 사용 가능: {len(list(correct_model.parameters())) > 0}")

print("\n" + "=" * 60)
print("[ 잘못된 모델 ]")
print("=" * 60)
try:
    incorrect_model = IncorrectModel()
except AttributeError as e:
    print(f"✗ 에러 발생:")
    print(f"   {str(e)}")
    print(f"\n해석: super().__init__()을 호출해야만")
    print(f"     nn.Module의 초기화 시스템이 활성화되어")
    print(f"     서브모듈(self.linear)을 등록할 수 있습니다!")

print("\n" + "=" * 60)
print("[ 결론 ]")
print("=" * 60)
print("✓ super().__init__()을 반드시 호출해야")
print("  1. 부모 클래스(nn.Module)의 초기화 시스템 활성화")
print("  2. 서브모듈 등록 가능")
print("  3. 매개변수 관리, 역전파, 학습 등 모든 기능 정상 작동")
print("  4. optimizer.step()으로 가중치 업데이트 가능")

[ 올바른 모델 ]
✓ 모델 생성 성공
매개변수 목록: [Parameter containing:
tensor([[0.7388]], requires_grad=True), Parameter containing:
tensor([0.1354], requires_grad=True)]
model.parameters() 사용 가능: True

[ 잘못된 모델 ]
✗ 에러 발생:
   cannot assign module before Module.__init__() call

해석: super().__init__()을 호출해야만
     nn.Module의 초기화 시스템이 활성화되어
     서브모듈(self.linear)을 등록할 수 있습니다!

[ 결론 ]
✓ super().__init__()을 반드시 호출해야
  1. 부모 클래스(nn.Module)의 초기화 시스템 활성화
  2. 서브모듈 등록 가능
  3. 매개변수 관리, 역전파, 학습 등 모든 기능 정상 작동
  4. optimizer.step()으로 가중치 업데이트 가능


## super()와 생성자(__init__)의 완전한 이해

### 1단계: 클래스와 생성자(__init__) 기초

#### 클래스란?
```python
class 동물:                    # 클래스 정의
    def __init__(self, 이름):   # 생성자 (생성 시 자동 호출)
        self.이름 = 이름
```

- **클래스**: 객체를 만들기 위한 설계도/틀
- **`__init__`**: 클래스로부터 객체가 생성될 때 **자동으로 호출되는 메서드**
- **`self`**: 생성된 객체 자신을 가리키는 참조

#### 생성자 호출 흐름

```
dog = 동물("뽀삐")  →  자동으로 __init__("뽀삐") 호출  →  dog.이름 = "뽀삐"
```

---

### 2단계: 상속(Inheritance) 이해하기

#### 상속이란?
부모 클래스의 특성/메서드를 자식 클래스가 물려받는 것

```python
class 동물:                      # 부모 클래스
    def __init__(self, 이름):
        self.이름 = 이름
        print("동물이 생성되었습니다")

class 개(동물):                   # 자식 클래스 (동물을 상속)
    def __init__(self, 이름, 견종):
        # 부모의 초기화는?? 어떻게 해줄까?
        self.이름 = 이름
        self.견종 = 견종
```

문제: 부모 클래스의 `__init__`도 호출해야 하는데, 코드가 중복됩니다.

---

### 3단계: super()의 역할

`super()`는 **"부모 클래스의 메서드를 호출해주는 도구"**

```python
class 개(동물):
    def __init__(self, 이름, 견종):
        super(개, self).__init__(이름)  # 부모의 __init__ 호출!
        self.견종 = 견종
```

이제 부모의 초기화 로직을 다시 쓸 필요가 없습니다.

---

### 4단계: super()의 정확한 문법 분석

`super(CorrectModel, self).__init__()`을 분해하면:

```
┌─────────────────────────────────────────────────────┐
│ super(CorrectModel, self).__init__()                │
├─────────────────────────────────────────────────────┤
│                                                     │
│  super()의 인자:                                     │
│  ├─ CorrectModel: 현재 클래스                       │
│  └─ self: 현재 객체 자신                            │
│                                                     │
│  반환값: 부모 클래스(nn.Module)의 프록시 객체       │
│  ↓                                                  │
│  .__init__(): 부모 클래스의 생성자 메서드 호출     │
│                                                     │
└─────────────────────────────────────────────────────┘
```

---

### 5단계: 상속 체인에서 작동하는 과정

```
프로그래머가 작성한 코드:

class CorrectModel(nn.Module):  # 상속 선언
    def __init__(self):
        super(CorrectModel, self).__init__()  # ← 여기서 뭐가 일어나나?
        self.linear = nn.Linear(1, 1)

실행 순서:
1. CorrectModel() 호출
   ↓
2. CorrectModel.__init__(self) 실행 시작
   ↓
3. super(CorrectModel, self).__init__() 만남
   ↓
4. nn.Module.__init__(self) 호출 (부모의 생성자)
   ↓
5. nn.Module의 초기화 로직 완료
   (매개변수 관리 시스템, 역전파 메커니즘 등 설정)
   ↓
6. CorrectModel.__init__()로 돌아와서 계속 실행
   ↓
7. self.linear = nn.Linear(1, 1) 실행 가능
   (이제 부모의 초기화가 완료되었으므로!)
   ↓
8. CorrectModel 객체 생성 완료
```

---

### 6단계: 메서드 오버라이딩 개념

자식 클래스가 부모 클래스의 메서드를 덮어씌우는 것:

```python
class 동물:
    def 울다(self):
        print("어? 어?")

class 개(동물):
    def 울다(self):  # 부모 메서드를 덮어씌움
        print("멍! 멍!")

dog = 개()
dog.울다()  # 출력: "멍! 멍!" (개의 울다 메서드가 실행됨)
```

PyTorch 예시:
- `nn.Module`은 `forward()` 메서드를 정의함
- 우리는 `forward()`를 오버라이딩해서 자신만의 신경망 로직을 작성

---

### 핵심 요약

| 개념 | 의미 |
|------|------|
| `__init__` | 클래스가 객체를 만들 때 자동 호출되는 생성자 메서드 |
| `self` | 지금 만들어지고 있는 객체 자신 |
| `상속` | 부모 클래스의 특성/메서드를 자식이 물려받음 |
| `super(자식클래스, self)` | "부모 클래스의 메서드에 접근하는 도구" 반환 |
| `super().__init__()` | 부모의 생성자 메서드를 호출 |
| `오버라이딩` | 부모의 메서드를 자식이 다시 정의해서 덮어씌움 |

---

### Python 3.6+ 간단한 표기법

```python
# 예전 방식 (Python 2/초기 Python 3)
super(CorrectModel, self).__init__()

# 간단한 방식 (Python 3.6+)
super().__init__()
```

둘 다 같은 의미이며, 최근에는 `super().__init__()`를 많이 씁니다.

In [8]:
print("=" * 70)
print("[ 기초 예제 1: 생성자(__init__)와 self 이해하기 ]")
print("=" * 70)

class 동물:
    def __init__(self, 이름):
        print(f"  [동물.__init__ 실행]")
        self.이름 = 이름
        print(f"  → self.이름 = '{이름}' 설정됨")

print("\n개 객체 생성:")
개 = 동물("뽀삐")
print(f"결과: 개.이름 = '{개.이름}'")

print("\n" + "=" * 70)
print("[ 기초 예제 2: 상속과 super() ]")
print("=" * 70)

class 개(동물):  # 동물을 상속받음
    def __init__(self, 이름, 견종):
        print(f"  [개.__init__ 시작]")
        print(f"  → super() 호출 전")
        
        # 부모 클래스의 __init__을 명시적으로 호출
        super(개, self).__init__(이름)
        
        print(f"  → super() 호출 후 (부모 초기화 완료!)")
        self.견종 = 견종
        print(f"  → self.견종 = '{견종}' 설정됨")
        print(f"  [개.__init__ 종료]")

print("\n말티즈 객체 생성:")
말티즈 = 개("뽀삐", "말티즈")
print(f"\n결과:")
print(f"  말티즈.이름 = '{말티즈.이름}' (부모 클래스에서 설정)")
print(f"  말티즈.견종 = '{말티즈.견종}' (자신 클래스에서 설정)")

print("\n" + "=" * 70)
print("[ 핵심 예제 3: super()가 없으면 뭐가 빠질까? ]")
print("=" * 70)

class 개_비정상(동물):
    def __init__(self, 이름, 견종):
        print(f"  [개_비정상.__init__ 시작]")
        # ⚠️ super().__init__() 호출 안 함!
        self.견종 = 견종
        print(f"  [개_비정상.__init__ 종료]")

print("\n비정상 말티즈 객체 생성:")
비정상_말티즈 = 개_비정상("뽀삐", "말티즈")

print(f"\n결과:")
print(f"  비정상_말티즈.견종 = '{비정상_말티즈.견종}' (있음)")
try:
    print(f"  비정상_말티즈.이름 = '{비정상_말티즈.이름}'")
except AttributeError:
    print(f"  ✗ AttributeError: 비정상_말티즈.이름 속성이 없음!")
    print(f"    → 부모의 __init__을 호출하지 않아서 self.이름이 설정 안 됨")

print("\n" + "=" * 70)
print("[ PyTorch 실제 예제: nn.Module 상속 ]")
print("=" * 70)

import torch
import torch.nn as nn

class 올바른_신경망(nn.Module):
    def __init__(self):
        print(f"  [올바른_신경망.__init__ 시작]")
        
        # ✓ 부모(nn.Module)의 초기화 필수!
        super(올바른_신경망, self).__init__()
        print(f"  → super().__init__() 호출 완료")
        
        # 이제 부모의 초기화가 완료되어 서브모듈 등록 가능
        self.linear = nn.Linear(2, 1)
        print(f"  → self.linear 등록 완료")

print("\n신경망 생성:")
model = 올바른_신경망()

print(f"\n부모(nn.Module)에서 제공하는 메서드들:")
print(f"  ✓ model.parameters() 사용 가능: {len(list(model.parameters())) > 0}")
print(f"  ✓ model.train() / model.eval() 사용 가능")
print(f"  ✓ model.to(device) 사용 가능")
print(f"  → 모두 부모의 __init__이 이런 것들을 초기화했기 때문!")

print("\n" + "=" * 70)
print("[ 메서드 오버라이딩 ]")
print("=" * 70)

class 동물_울기:
    def 울다(self):
        return "어? 어?"

class 개_울기(동물_울기):
    def 울다(self):  # 부모 메서드를 덮어씌움
        return "멍! 멍!"

class 고양이_울기(동물_울기):
    def 울다(self):  # 부모 메서드를 덮어씌움
        return "야옹! 야옹!"

동물1 = 동물_울기()
개1 = 개_울기()
고양이1 = 고양이_울기()

print(f"동물.울다() = '{동물1.울다()}' (부모의 메서드)")
print(f"개.울다() = '{개1.울다()}' (자식이 오버라이딩한 메서드)")
print(f"고양이.울다() = '{고양이1.울다()}' (자식이 오버라이딩한 메서드)")

print("\nPyTorch 맥락:")
print(f"  nn.Module.forward() ← 부모가 정의 (아무것도 안 함)")
print(f"  MyModel.forward() ← 자식이 오버라이딩 (우리가 정의한 신경망 로직)")

print("\n" + "=" * 70)
print("✓ 정리: super(클래스명, self).__init__()의 역할")
print("=" * 70)
print("""
1. super(클래스명, self) → 부모 클래스에 접근하는 프록시 객체 생성
2. .__init__() → 부모 클래스의 생성자 메서드 호출
3. 결과: 부모의 초기화 로직 완료
   → 매개변수 관리, 역전파 메커니즘, 메서드 상속 등 모두 준비됨
4. 그 다음에 자식 클래스의 초기화 로직 계속 진행
""")

[ 기초 예제 1: 생성자(__init__)와 self 이해하기 ]

개 객체 생성:
  [동물.__init__ 실행]
  → self.이름 = '뽀삐' 설정됨
결과: 개.이름 = '뽀삐'

[ 기초 예제 2: 상속과 super() ]

말티즈 객체 생성:
  [개.__init__ 시작]
  → super() 호출 전
  [동물.__init__ 실행]
  → self.이름 = '뽀삐' 설정됨
  → super() 호출 후 (부모 초기화 완료!)
  → self.견종 = '말티즈' 설정됨
  [개.__init__ 종료]

결과:
  말티즈.이름 = '뽀삐' (부모 클래스에서 설정)
  말티즈.견종 = '말티즈' (자신 클래스에서 설정)

[ 핵심 예제 3: super()가 없으면 뭐가 빠질까? ]

비정상 말티즈 객체 생성:
  [개_비정상.__init__ 시작]
  [개_비정상.__init__ 종료]

결과:
  비정상_말티즈.견종 = '말티즈' (있음)
  ✗ AttributeError: 비정상_말티즈.이름 속성이 없음!
    → 부모의 __init__을 호출하지 않아서 self.이름이 설정 안 됨

[ PyTorch 실제 예제: nn.Module 상속 ]

신경망 생성:
  [올바른_신경망.__init__ 시작]
  → super().__init__() 호출 완료
  → self.linear 등록 완료

부모(nn.Module)에서 제공하는 메서드들:
  ✓ model.parameters() 사용 가능: True
  ✓ model.train() / model.eval() 사용 가능
  ✓ model.to(device) 사용 가능
  → 모두 부모의 __init__이 이런 것들을 초기화했기 때문!

[ 메서드 오버라이딩 ]
동물.울다() = '어? 어?' (부모의 메서드)
개.울다() = '멍! 멍!' (자식이 오버라이딩한 메서드)
고양이.울다() = '야옹! 야옹!' (자식이 오버라이딩한 메서드)

PyTorch 맥락:
  nn.Module.forward